# ProtT5 — Per-Residue Embedding Extraction

**PP1 SoSe2026 | TU München**
Team: Joel Simon, Julius Schmidt, Lisa Börner, Andreas Weitz

Inferenz-Notebook: zieht **per-Residue** ProtT5-Embeddings (`[L, 1024]` float32) für die FLIP-Datasets und schreibt sie als HDF5 nach Google Drive.

Voraussetzung: `colab_embedding_extraction.ipynb` wurde einmal ausgeführt und hat die FASTA-Splits unter `/content/drive/MyDrive/pp1_sop/raw/{deeploc,meltome}/` abgelegt.

HDF5-Layout (kompatibel mit `src/sop/data/store.py::EmbeddingStore`):
```
/{seq_id}/embeddings        float32 [L, 1024]
/{seq_id}.attrs['length']   int
```

### Reihenfolge
1. GPU Check
2. Drive mounten
3. Configuration
4. Dependencies
5. Modell laden
6. Extraction-Funktion
7. Testlauf (100 Sequenzen)
8. Voller Run — 6 Splits
9. Output prüfen


## 1 · GPU Check

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        'No GPU found. Runtime -> Change runtime type -> GPU and restart.'
    )

gpu = torch.cuda.get_device_properties(0)
print(f'GPU:  {gpu.name}')
print(f'VRAM: {gpu.total_memory / 1e9:.1f} GB')
print(f'CUDA: {torch.version.cuda}')


## 2 · Google Drive mounten

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
print('Drive mounted. Top-level folders:')
print(os.listdir('/content/drive/MyDrive'))


## 3 · Configuration

Pfade und Modell. Standardmäßig nichts anpassen — die FASTA-Files liegen schon auf Drive (vom anderen Notebook erzeugt).


In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/pp1_sop')
RAW_ROOT   = DRIVE_ROOT / 'raw'
EMBED_OUT  = DRIVE_ROOT / 'embeddings'

# ProtT5 encoder von HuggingFace (Rostlab-Empfehlung: half-precision encoder-only)
MODEL_ID   = 'Rostlab/prot_t5_xl_half_uniref50-enc'

BATCH_SIZE = 4      # bei CUDA OOM auf 2 oder 1 reduzieren
DEVICE     = 'cuda'

EMBED_OUT.mkdir(parents=True, exist_ok=True)

# Sanity-Check: alle 6 FASTA-Dateien vorhanden?
expected_fastas = [
    RAW_ROOT / 'deeploc' / 'train.fasta',
    RAW_ROOT / 'deeploc' / 'val.fasta',
    RAW_ROOT / 'deeploc' / 'test.fasta',
    RAW_ROOT / 'meltome' / 'train.fasta',
    RAW_ROOT / 'meltome' / 'val.fasta',
    RAW_ROOT / 'meltome' / 'test.fasta',
]
missing = [p for p in expected_fastas if not p.exists()]
if missing:
    raise FileNotFoundError(
        'FASTA-Dateien fehlen — bitte erst colab_embedding_extraction.ipynb laufen lassen:\n'
        + '\n'.join(f'  {p}' for p in missing)
    )

print(f'Modell:  {MODEL_ID}')
print(f'Output:  {EMBED_OUT}')
print(f'Input:   {RAW_ROOT}  ({len(expected_fastas)} FASTA-Dateien gefunden)')


## 4 · Dependencies installieren

In [ ]:
import subprocess, sys

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'transformers>=4.38', 'sentencepiece', 'h5py'],
    check=True,
)
print('Dependencies installiert.')


## 5 · ProtT5-Encoder laden

Erster Aufruf lädt ~3 GB von HuggingFace.

In [ ]:
import torch
from transformers import T5EncoderModel, T5Tokenizer

print(f'Lade Tokenizer ({MODEL_ID})...')
tokenizer = T5Tokenizer.from_pretrained(MODEL_ID, do_lower_case=False)

print('Lade Modell (~3 GB beim ersten Mal)...')
model = T5EncoderModel.from_pretrained(MODEL_ID)
model.eval().to(DEVICE)
for p in model.parameters():
    p.requires_grad_(False)

n_layers = model.config.num_layers
d_model  = model.config.d_model
print(f'Layers: {n_layers}  |  Hidden dim d: {d_model}')

# Smoke-Test
test_seq = 'MKTLLLTLVVVTIVCLDLGAVGNK'
enc = tokenizer([' '.join(list(test_seq))], add_special_tokens=True,
                return_tensors='pt').to(DEVICE)
with torch.inference_mode():
    out = model(**enc)
emb = out.last_hidden_state[0, :len(test_seq), :]
print(f'Smoke-Test OK: L={len(test_seq)} -> Embedding {tuple(emb.shape)}')
assert emb.shape == (len(test_seq), d_model)

del enc, out, emb
torch.cuda.empty_cache()


## 6 · Extraction-Funktion

`extract_to_hdf5()`:
- FASTA einlesen
- Nach Sequenzlänge sortieren (kompakteres Padding → schnellere Inferenz)
- Batch-weise durch ProtT5 forward-passen
- Per-Residue-Embeddings (`[L, 1024]` float32) ins HDF5 schreiben
- Bei CUDA OOM: Fallback auf single-sequence batch

HDF5-Layout: `/{seq_id}/embeddings` Dataset + `length` Attribut → kompatibel mit `EmbeddingStore`.

Standard-AA-Mapping wie in der Original-Rostlab-Inferenz: U/Z/O → X (rare/ambiguous Residues, sonst stolpert der Tokenizer).


In [ ]:
import h5py
import torch
from pathlib import Path


def parse_fasta(path):
    records, seq_id, parts = [], None, []
    with open(path) as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            if line.startswith('>'):
                if seq_id is not None:
                    records.append((seq_id, ''.join(parts)))
                seq_id = line[1:].split()[0].replace('/', '_').replace('.', '_')
                parts = []
            else:
                parts.append(line.upper().replace('-', ''))
    if seq_id is not None:
        records.append((seq_id, ''.join(parts)))
    # rare/ambiguous AAs auf X mappen
    cleaned = [(sid, s.replace('U', 'X').replace('Z', 'X').replace('O', 'X'))
               for sid, s in records]
    return cleaned


@torch.inference_mode()
def _embed_batch(seqs):
    spaced = [' '.join(list(s)) for s in seqs]
    enc = tokenizer(spaced, add_special_tokens=True, padding=True,
                    return_tensors='pt').to(DEVICE)
    out = model(**enc).last_hidden_state          # [B, T, d]
    # ProtT5: kein BOS, ein EOS am Ende -> erste L Positionen pro Sequenz nehmen
    return [out[i, :len(s), :].detach().cpu().float() for i, s in enumerate(seqs)]


def extract_to_hdf5(fasta_path, output_path, batch_size=BATCH_SIZE):
    fasta_path  = Path(fasta_path)
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    records = parse_fasta(fasta_path)
    records.sort(key=lambda r: len(r[1]))
    n = len(records)
    print(f'  {fasta_path.name}: {n} Sequenzen | Laengen {len(records[0][1])}-{len(records[-1][1])}')

    n_written = 0
    with h5py.File(output_path, 'w') as fh:
        next_log = 200
        for start in range(0, n, batch_size):
            batch = records[start : start + batch_size]
            ids, seqs = zip(*batch)
            try:
                embs = _embed_batch(list(seqs))
            except torch.cuda.OutOfMemoryError:
                print(f'  OOM bei batch start={start}, fallback auf single')
                torch.cuda.empty_cache()
                embs = []
                for sid, s in zip(ids, seqs):
                    try:
                        embs.append(_embed_batch([s])[0])
                    except torch.cuda.OutOfMemoryError:
                        print(f'    SKIP {sid} (L={len(s)}) - too long')
                        torch.cuda.empty_cache()
                        embs.append(None)

            for sid, emb in zip(ids, embs):
                if emb is None:
                    continue
                arr = emb.numpy()
                grp = fh.require_group(sid)
                grp.create_dataset('embeddings', data=arr,
                                   compression='lzf', chunks=True)
                grp.attrs['length'] = arr.shape[0]
                n_written += 1

            done = min(start + batch_size, n)
            if done >= next_log or done == n:
                print(f'    {done}/{n} verarbeitet, {n_written} geschrieben')
                next_log += 200

    size_mb = output_path.stat().st_size / 1e6
    print(f'  -> {output_path} ({size_mb:.1f} MB, {n_written}/{n} Proteine)\n')


def run_extraction(fasta, output, batch_size=BATCH_SIZE):
    print(f'Starte: {Path(fasta).name} -> {Path(output).name}')
    extract_to_hdf5(fasta, output, batch_size=batch_size)


## 7 · Testlauf mit 100 Sequenzen

Immer zuerst — bevor der volle Datensatz läuft.

In [ ]:
import textwrap
from pathlib import Path

def subsample_fasta(src: Path, dst: Path, n: int = 100):
    records, seq_id, parts = [], None, []
    with open(src) as fh:
        for line in fh:
            line = line.rstrip()
            if not line:
                continue
            if line.startswith('>'):
                if seq_id is not None:
                    records.append((seq_id, ''.join(parts)))
                seq_id, parts = line, []
            else:
                parts.append(line)
    if seq_id is not None:
        records.append((seq_id, ''.join(parts)))
    records = records[:n]
    dst.parent.mkdir(parents=True, exist_ok=True)
    with open(dst, 'w') as fh:
        for header, seq in records:
            fh.write(header + '\n')
            for chunk in textwrap.wrap(seq, 60):
                fh.write(chunk + '\n')
    print(f'Subsampled {len(records)} Sequenzen -> {dst}')

SCRATCH     = Path('/content/test_run')
test_fasta  = SCRATCH / 'deeploc_100.fasta'
test_output = SCRATCH / 'deeploc_100.h5'

subsample_fasta(RAW_ROOT / 'deeploc' / 'train.fasta', test_fasta, n=100)


In [ ]:
run_extraction(fasta=test_fasta, output=test_output, batch_size=4)


In [ ]:
import h5py

with h5py.File(test_output, 'r') as f:
    ids  = list(f.keys())
    emb0 = f[ids[0]]['embeddings'][:]
    L0   = f[ids[0]].attrs['length']

print(f'Proteine im Test-HDF5: {len(ids)}')
print(f'Erstes Embedding: id={ids[0]!r}  shape={emb0.shape}  dtype={emb0.dtype}  length-attr={L0}')
assert emb0.ndim == 2 and emb0.shape[1] == 1024, 'Unerwartete Embedding-Dimension!'
assert emb0.shape[0] == L0, 'Length-Attribut stimmt nicht mit shape ueberein!'
print('Testlauf erfolgreich.')


## 8 · Voller Run — alle 6 Splits

Output direkt nach Drive (`pp1_sop/embeddings/`).

**Typische T4-Laufzeiten:**
- DeepLoc train (~9.5k): ~15-20 min
- DeepLoc val (~1.7k): ~3-4 min
- DeepLoc test (~2.8k): ~5-6 min
- Meltome train (~22k): ~40-50 min
- Meltome val (~2.5k): ~5 min
- Meltome test (~3.1k): ~6 min

> Bei `CUDA out of memory`: `BATCH_SIZE` in Zelle 3 auf 2 oder 1 setzen und ab hier neu starten.


In [ ]:
run_extraction(
    fasta  = RAW_ROOT / 'deeploc' / 'train.fasta',
    output = EMBED_OUT / 'deeploc_train.h5',
)


In [ ]:
run_extraction(
    fasta  = RAW_ROOT / 'deeploc' / 'val.fasta',
    output = EMBED_OUT / 'deeploc_val.h5',
)


In [ ]:
run_extraction(
    fasta  = RAW_ROOT / 'deeploc' / 'test.fasta',
    output = EMBED_OUT / 'deeploc_test.h5',
)


In [ ]:
run_extraction(
    fasta  = RAW_ROOT / 'meltome' / 'train.fasta',
    output = EMBED_OUT / 'meltome_train.h5',
)


In [ ]:
run_extraction(
    fasta  = RAW_ROOT / 'meltome' / 'val.fasta',
    output = EMBED_OUT / 'meltome_val.h5',
)


In [ ]:
run_extraction(
    fasta  = RAW_ROOT / 'meltome' / 'test.fasta',
    output = EMBED_OUT / 'meltome_test.h5',
)


## 9 · Output prüfen

Danach lokal aus Drive runterladen nach `data/embeddings/` und mit den Repo-Skripten weiterarbeiten.

In [ ]:
import h5py

expected = [
    'deeploc_train.h5', 'deeploc_val.h5', 'deeploc_test.h5',
    'meltome_train.h5', 'meltome_val.h5', 'meltome_test.h5',
]

print(f'{"Datei":<30} {"Vorhanden":<12} {"Proteine":<12} {"Groesse (MB)"}')
print('-' * 65)
for fname in expected:
    p = EMBED_OUT / fname
    if p.exists():
        size_mb = p.stat().st_size / 1e6
        with h5py.File(p, 'r') as f:
            n = len(f.keys())
        print(f'{fname:<30} {"JA":<12} {n:<12} {size_mb:.1f}')
    else:
        print(f'{fname:<30} {"FEHLT":<12}')
